# Lecture 06: Preparation for the Local Theory

**Source span.** Printed pages 33-39; physical PDF pages 47-53 in `Lectures on Symplectic Geometry.pdf` according to the course source map. I also checked the local PDF text for the lecture window before revising this notebook.

**Lecture goal.** This lecture supplies the non-symplectic machinery used by the next local-normal-form theorems: isotopies and their time-dependent vector fields, Lie derivatives, tubular neighborhoods, deformation retracts, and the de Rham homotopy formula. The point is not to memorize a list of background theorems; it is to see why Moser-style arguments can replace a global problem by a controlled flow and a local exactness statement.

Read the chapter as a pipeline. A moving family of diffeomorphisms gives a vector field. The vector field differentiates pullbacks through the Lie derivative. A tubular neighborhood replaces a submanifold by its normal bundle. The radial contraction `(x, v) -> (x, t v)` then builds a homotopy operator, and the homotopy formula turns closed forms that vanish on the core into exact forms. That exact primitive is the ingredient Lecture 7 feeds into the Moser equation.

In [ ]:
from pathlib import Path
import json
import sys

import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import sympy as sp


def find_book_root(start=None):
    start = Path.cwd() if start is None else Path(start)
    for candidate in [start, *start.parents]:
        if (candidate / "AGENTS.md").exists() and (candidate / "Lectures on Symplectic Geometry.pdf").exists():
            return candidate
    raise RuntimeError("Could not locate the Lectures-on-Symplectic-Geometry course root.")


BOOK_ROOT = find_book_root()
if str(BOOK_ROOT) not in sys.path:
    sys.path.insert(0, str(BOOK_ROOT))

from utils.artifacts import display_artifact, save_json, save_matplotlib

ARTIFACT_TOPIC = "lecture-06"
ARTIFACT_ROOT = BOOK_ROOT / "artifacts" / ARTIFACT_TOPIC
FIG_DIR = ARTIFACT_ROOT / "figures"
CHECK_DIR = ARTIFACT_ROOT / "checks"
FIG_DIR.mkdir(parents=True, exist_ok=True)
CHECK_DIR.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({"figure.dpi": 130, "font.size": 10})
print(f"Book root: {BOOK_ROOT}")
print(f"Artifact root: {ARTIFACT_ROOT.relative_to(BOOK_ROOT)}")

## Translation Guide

The lecture sections translate into computational objects this way.

| Source idea | Computational object | What to inspect |
| --- | --- | --- |
| Isotopy `rho_t` | a time-indexed map and its generating vector field | arrows should be tangent to the visible motion |
| Lie derivative | derivative of a pullback along the flow | a finite-difference derivative should match `L_v` at small time |
| Tubular neighborhood | normal fibers over a submanifold | every nearby point should project to one base point while the radius is small |
| Deformation retract | radial contraction `(x, v) -> (x, t v)` | the core `X` stays fixed and normal coordinates shrink |
| Homotopy formula | `Id - rho_0^* = dQ + Qd` | exact symbolic residuals should vanish in a toy model |

## Library Routing

This lecture is mostly local differential topology, so the notebook uses `matplotlib` for durable flow, fiber, and proof diagrams; `networkx` for the proof-dependency graph; `sympy` for an exact homotopy-operator check; and `numpy` for sample grids and residual values. A 3D mesh library would add weight without clarifying the local normal-bundle and flow mechanics here.

## Visual Storyboard

The visual sequence is chapter-specific.

1. **Isotopy and vector field.** A deformed grid and arrows show why `d rho_t / dt = v_t o rho_t` is a moving-coordinate statement, not just a derivative symbol.
2. **Tubular neighborhood and failure mode.** A circle has a safe annular normal neighborhood when the normal radius is small; the center is a visible counterexample when the radius is too large, because it has many nearest points.
3. **Proof dependency graph.** The local theory depends on Cartan's formula, the derivative of pullbacks, the tubular retract, and the homotopy operator.
4. **Exact homotopy check.** On coordinates `(x, v)` in a normal bundle, the radial contraction verifies the homotopy formula for a function and a one-form exactly.

The artifacts are saved under `artifacts/lecture-06/` and are regenerated by the code cells below.

In [ ]:
# Isotopy visual: a local flow deforms points while arrows show the generating field.
x = np.linspace(-1.4, 1.4, 13)
y = np.linspace(-1.0, 1.0, 11)
X, Y = np.meshgrid(x, y)
points = np.column_stack([X.ravel(), Y.ravel()])

def vector_field(p):
    x, y = p[..., 0], p[..., 1]
    return np.stack([-0.35 * y, 0.55 * x], axis=-1)

# First-order flow approximation, enough to visualize the local bookkeeping.
t = 0.55
V = vector_field(points)
flowed = points + t * V

fig, ax = plt.subplots(figsize=(7.2, 4.6))
for yy in y:
    row = points[np.isclose(points[:, 1], yy)]
    row_flow = row + t * vector_field(row)
    ax.plot(row[:, 0], row[:, 1], color="#c7d2fe", lw=0.8)
    ax.plot(row_flow[:, 0], row_flow[:, 1], color="#4f46e5", lw=1.2)
for xx in x:
    col = points[np.isclose(points[:, 0], xx)]
    col_flow = col + t * vector_field(col)
    ax.plot(col[:, 0], col[:, 1], color="#c7d2fe", lw=0.8)
    ax.plot(col_flow[:, 0], col_flow[:, 1], color="#4f46e5", lw=1.2)
ax.quiver(points[::8, 0], points[::8, 1], V[::8, 0], V[::8, 1], color="#dc2626", angles="xy", scale_units="xy", scale=5)
ax.set_title("Isotopy data: grid motion plus generating vector field")
ax.set_aspect("equal")
ax.set_xlabel("local coordinate x")
ax.set_ylabel("local coordinate y")
ax.text(-1.35, 0.9, "p moves by rho_t", color="#4f46e5")
ax.text(0.35, -0.85, "v_t is read at the moved point", color="#dc2626")
flow_path = save_matplotlib(fig, ARTIFACT_TOPIC, "figures", "isotopy-flow-lie-derivative.png")
plt.close(fig)
display_artifact(flow_path, width=760)
flow_path.relative_to(BOOK_ROOT).as_posix()

In [ ]:
# Tubular neighborhood visual: safe projection and the too-large-radius failure mode.
theta = np.linspace(0, 2 * np.pi, 400)
circle = np.column_stack([np.cos(theta), np.sin(theta)])
r_safe = 0.22
outer = (1 + r_safe) * circle
inner = (1 - r_safe) * circle
sample_theta = np.linspace(0, 2 * np.pi, 16, endpoint=False)
base = np.column_stack([np.cos(sample_theta), np.sin(sample_theta)])
normal_end = 1.28 * base
bad_point = np.array([0.0, 0.0])

fig, ax = plt.subplots(figsize=(6.4, 6.0))
ax.fill(outer[:, 0], outer[:, 1], color="#ecfeff", alpha=0.95, label="safe normal band")
ax.fill(inner[:, 0], inner[:, 1], color="white")
ax.plot(circle[:, 0], circle[:, 1], color="#0f766e", lw=2.2, label="submanifold X")
for b, e in zip(base, normal_end):
    ax.plot([b[0], e[0]], [b[1], e[1]], color="#0891b2", lw=1.0)
ax.scatter(base[:, 0], base[:, 1], s=20, color="#0f766e")
ax.scatter([bad_point[0]], [bad_point[1]], s=90, color="#dc2626", marker="x", label="nonunique projection if radius reaches center")
for angle in np.linspace(0, 2*np.pi, 6, endpoint=False):
    b = np.array([np.cos(angle), np.sin(angle)])
    ax.plot([0, b[0]], [0, b[1]], color="#fecaca", lw=0.9, ls="--")
ax.set_title("Tubular neighborhood: normal fibers are useful only before projection stops being unique")
ax.set_aspect("equal")
ax.axis("off")
ax.legend(loc="upper right")
tube_path = save_matplotlib(fig, ARTIFACT_TOPIC, "figures", "tubular-neighborhood-projection-failure.png")
plt.close(fig)
display_artifact(tube_path, width=660)
tube_path.relative_to(BOOK_ROOT).as_posix()

In [ ]:
# Proof dependency graph for the local-theory pipeline.
edges = [
    ("isotopy", "time-dependent vector field"),
    ("time-dependent vector field", "Lie derivative"),
    ("Lie derivative", "derivative of pullbacks"),
    ("Cartan formula", "derivative of pullbacks"),
    ("tubular neighborhood", "radial retraction"),
    ("radial retraction", "homotopy operator Q"),
    ("derivative of pullbacks", "homotopy formula"),
    ("homotopy operator Q", "homotopy formula"),
    ("homotopy formula", "closed form becomes exact"),
    ("closed form becomes exact", "Moser primitive in Lecture 7"),
]
G = nx.DiGraph(edges)
pos = nx.spring_layout(G, seed=6, k=0.9)
fig, ax = plt.subplots(figsize=(9.2, 5.8))
nx.draw_networkx_edges(G, pos, ax=ax, arrows=True, arrowstyle="-|>", arrowsize=14, edge_color="#64748b", width=1.4)
nx.draw_networkx_nodes(G, pos, ax=ax, node_size=1700, node_color="#f8fafc", edgecolors="#334155", linewidths=1.0)
nx.draw_networkx_labels(G, pos, ax=ax, font_size=8)
ax.set_title("Proof dependency graph: why Lecture 6 is the entry ramp to Moser's trick")
ax.axis("off")
proof_path = save_matplotlib(fig, ARTIFACT_TOPIC, "figures", "homotopy-formula-proof-dependencies.png")
plt.close(fig)
display_artifact(proof_path, width=820)
proof_path.relative_to(BOOK_ROOT).as_posix()

## Homotopy Formula As A Computation

Work in a model normal bundle with coordinates `(x, v)` and core `X = {v = 0}`. The radial maps are `rho_t(x, v) = (x, t v)`. The vector tangent to the path `t -> rho_t(x, v)` is `v partial_v` at the point `(x, t v)`.

For a function `f(x, v) = x^2 + v^2`, the homotopy formula in degree zero says that `Q(df) = f - rho_0^* f`. For the one-form `alpha = v dx`, it says `Q(d alpha) = alpha - rho_0^* alpha`. These are small exact checks of the mechanism used in the lecture: the radial retract kills the normal direction and the operator `Q` records the missing normal part.

In [ ]:
x_sym, v_sym, t_sym = sp.symbols("x v t", real=True)
f = x_sym**2 + v_sym**2
# Q(df) = integral v * (partial f / partial v)(x, t v) dt
Q_df = sp.integrate(v_sym * sp.diff(f, v_sym).subs(v_sym, t_sym * v_sym), (t_sym, 0, 1))
f_minus_retract = f - f.subs(v_sym, 0)
function_residual = sp.simplify(Q_df - f_minus_retract)

# alpha = v dx.  d alpha = dv wedge dx.  Contracting with v partial_v gives v dx.
Q_dalpha_dx_coefficient = sp.integrate(v_sym, (t_sym, 0, 1))
alpha_minus_retract_dx_coefficient = v_sym - 0
one_form_residual = sp.simplify(Q_dalpha_dx_coefficient - alpha_minus_retract_dx_coefficient)

checks = {
    "function_Qdf": str(sp.simplify(Q_df)),
    "function_expected": str(sp.simplify(f_minus_retract)),
    "function_residual": str(function_residual),
    "one_form_Qdalpha_dx_coefficient": str(Q_dalpha_dx_coefficient),
    "one_form_expected_dx_coefficient": str(alpha_minus_retract_dx_coefficient),
    "one_form_residual": str(one_form_residual),
    "passed": function_residual == 0 and one_form_residual == 0,
}
check_path = save_json(checks, ARTIFACT_TOPIC, "checks", "homotopy-operator-residuals.json")
assert checks["passed"]
print(json.dumps(checks, indent=2))
print(check_path.relative_to(BOOK_ROOT).as_posix())

## Failure Mode: Why The Hypotheses Matter

The tubular-neighborhood theorem does not say that every thickening of a submanifold has a well-defined nearest-point projection. The circle picture marks the center as a failure point: it is equally close to every point of the circle, so `pi(p)` is not unique. This is the geometric reason the theorem works with a sufficiently small neighborhood, or with a radius function that shrinks in the noncompact case.

The homotopy formula has its own hidden warning. The radial contraction works because each fiber is convex and stays in the tubular neighborhood during the contraction. If a proposed neighborhood is not closed under `(x, v) -> (x, t v)`, the formula no longer describes a homotopy inside the space where the form is defined. That is exactly the sort of local-control issue Moser arguments must get right.

In [ ]:
source_span = {
    "course": "Lectures-on-Symplectic-Geometry",
    "lecture": 6,
    "title": "Preparation for the Local Theory",
    "printed_span": "33-39",
    "physical_pdf_span": "47-53",
    "source_file": "Lectures on Symplectic Geometry.pdf",
    "source_map_authority": "source-map.json and scripts/lsg_inventory.py",
    "source_terms_used": [
        "isotopy",
        "time-dependent vector field",
        "Lie derivative",
        "tubular neighborhood theorem",
        "normal bundle",
        "radial retraction",
        "homotopy operator",
        "Cartan formula",
    ],
}
source_path = save_json(source_span, ARTIFACT_TOPIC, "checks", "source-span.json")

storyboard = {
    "chapter_goal": "Use flows, tubular neighborhoods, and the homotopy formula as concrete local tools for later Moser normal-form arguments.",
    "library_routing": [
        {"concept": "isotopy and vector field", "library": "matplotlib + numpy", "why": "a local flow is best inspected as a deformed grid with arrows"},
        {"concept": "tubular neighborhood", "library": "matplotlib", "why": "normal fibers and projection failure are planar and need labels, not 3D rotation"},
        {"concept": "proof dependencies", "library": "networkx", "why": "the lecture is a chain of proof tools feeding the Moser trick"},
        {"concept": "homotopy formula", "library": "sympy", "why": "the residual can be checked exactly in a toy normal-bundle model"},
    ],
    "visual_sequence": [
        {"concept": "isotopy data", "artifact": "artifacts/lecture-06/figures/isotopy-flow-lie-derivative.png", "inspection_target": "arrows generate the visible grid motion"},
        {"concept": "tubular projection", "artifact": "artifacts/lecture-06/figures/tubular-neighborhood-projection-failure.png", "inspection_target": "small normal bands have unique projection; the center failure has many nearest points"},
        {"concept": "proof scaffold", "artifact": "artifacts/lecture-06/figures/homotopy-formula-proof-dependencies.png", "inspection_target": "Cartan formula plus pullback differentiation feed the homotopy formula"},
        {"concept": "homotopy residual", "artifact": "artifacts/lecture-06/checks/homotopy-operator-residuals.json", "inspection_target": "symbolic residuals vanish exactly"},
    ],
}
storyboard_path = save_json(storyboard, ARTIFACT_TOPIC, "checks", "visual-storyboard.json")
print(source_path.relative_to(BOOK_ROOT).as_posix())
print(storyboard_path.relative_to(BOOK_ROOT).as_posix())

## Learner Lab

Change the vector field in the first plot and ask which object is being differentiated: the point, the pullback, or the form itself. Then change the tubular-neighborhood radius in the second plot and identify where nearest-point projection first becomes ambiguous. Finally, replace `f = x^2 + v^2` in the symbolic cell with another polynomial and check that the degree-zero homotopy formula still returns `f(x, v) - f(x, 0)`.

**Takeaways.**

- An isotopy is paired with a time-dependent vector field by `d rho_t / dt = v_t o rho_t`; the vector field is read at the moved point.
- The derivative of a pulled-back time-dependent form needs both the Lie derivative term and the explicit `d omega_t / dt` term.
- A tubular neighborhood packages nearby points as normal directions over `X`, but only while projection remains unique.
- The radial contraction of the normal bundle gives the homotopy operator behind the exactness statement used in local Moser arguments.
- Lecture 7 uses this exactness mechanism to construct the primitive that solves the Moser equation.

In [ ]:
final_artifacts = [
    "artifacts/lecture-06/figures/isotopy-flow-lie-derivative.png",
    "artifacts/lecture-06/figures/tubular-neighborhood-projection-failure.png",
    "artifacts/lecture-06/figures/homotopy-formula-proof-dependencies.png",
    "artifacts/lecture-06/checks/homotopy-operator-residuals.json",
    "artifacts/lecture-06/checks/source-span.json",
    "artifacts/lecture-06/checks/visual-storyboard.json",
]

homotopy_checks = json.loads((BOOK_ROOT / "artifacts/lecture-06/checks/homotopy-operator-residuals.json").read_text(encoding="utf-8"))
assert homotopy_checks["passed"] is True
assert homotopy_checks["function_residual"] == "0"
assert homotopy_checks["one_form_residual"] == "0"

for relative in final_artifacts:
    path = BOOK_ROOT / relative
    assert path.exists(), f"missing artifact: {relative}"
    assert path.stat().st_size > 0, f"empty artifact: {relative}"

final_sanity = {
    "lecture": 6,
    "source_span": "printed pages 33-39; physical PDF pages 47-53",
    "artifacts": final_artifacts,
    "homotopy_checks": homotopy_checks,
    "passed": True,
}
final_path = save_json(final_sanity, ARTIFACT_TOPIC, "checks", "final-sanity.json")
print({
    "checked_artifact_count": len(final_artifacts),
    "homotopy_passed": homotopy_checks["passed"],
    "final_sanity": final_path.relative_to(BOOK_ROOT).as_posix(),
})